In [1]:
import random

import numpy as np
import torch
from torch.utils.data import DataLoader

In [2]:
from coronary_analysis.models.segmentation import (
    CoronaryUNetPP,
    CoronaryDeeplabV3Plus,
    CoronaryUNetCustom,
)
from coronary_analysis.transforms import (
    get_train_transforms,
    get_val_transforms,
)
from coronary_analysis.datasets import ArcadeSyntaxBinaryDataset
from coronary_analysis.metrics import BCEDiceClDiceCriterion
from coronary_analysis.train import training_loop

In [3]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

device

device(type='cuda')

In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    elif device.type == "mps":
        torch.mps.manual_seed(seed)

In [5]:
set_seed(42)

In [6]:
IMG_SIZE = 320

In [7]:
IMAGE_DIR = "raw_datasets/arcade/arcade/syntax"

arcade_train_ds = ArcadeSyntaxBinaryDataset(
    root=IMAGE_DIR,
    split="train",
    transform=get_train_transforms(IMG_SIZE),
)
arcade_val_ds = ArcadeSyntaxBinaryDataset(
    root=IMAGE_DIR,
    split="val",
    transform=get_val_transforms(IMG_SIZE),
)

loading annotations into memory...
Done (t=0.18s)
creating index...
index created!
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!


In [ ]:
configs = [
    {
        "model": CoronaryUNetCustom(
            "resnet34", encoder_weights="imagenet", dropout=0.2
        ),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
    {
        "model": CoronaryUNetCustom(
            "resnet34", encoder_weights=None, dropout=0.3, depth=5
        ),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
    {
        "model": CoronaryUNetCustom(
            "resnet34", encoder_weights=None, dropout=0.2, depth=4
        ),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
    {
        "model": CoronaryUNetCustom(
            "resnet18", encoder_weights=None, dropout=0.2, depth=4
        ),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 100,
    },
    {
        "model": CoronaryUNetPP("resnet18", encoder_weights="imagenet"),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
    {
        "model": CoronaryUNetPP("resnet34", encoder_weights="imagenet"),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
    {
        "model": CoronaryDeeplabV3Plus("resnet18", encoder_weights="imagenet"),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
    {
        "model": CoronaryDeeplabV3Plus("resnet18", encoder_weights=None),
        "optimizer": torch.optim.Adam,
        "criterion": BCEDiceClDiceCriterion(0.5, 0.5, 0.0),
        "lr": 1e-3,
        "batch_size": 32,
        "num_epochs": 50,
    },
]

In [9]:
import os

PRETRAINED_MODELS_DIR = "../models/pretrained/arcade_syntax_binary"

if not os.path.exists(PRETRAINED_MODELS_DIR):
    os.makedirs(PRETRAINED_MODELS_DIR)

In [ ]:
for i, config in enumerate(configs):
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()

    model = config["model"].to(device)
    optimizer = config["optimizer"](model.parameters(), lr=config["lr"])
    criterion = config["criterion"]
    batch_size = config["batch_size"]
    num_epochs = config["num_epochs"]

    train_loader = DataLoader(arcade_train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(arcade_val_ds, batch_size=batch_size, shuffle=False)

    training_loop(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        num_epochs=num_epochs,
    )

    model_name = type(config["model"]).__name__ + "_" + str(i + 8)
    torch.save(model.state_dict(), f"{PRETRAINED_MODELS_DIR}/{model_name}.pth")